**Pipeline start here :**

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

**Prompt Variations**

In [ ]:
CRACK_PROMPTS = [
    "segment crack",
    "segment wall crack"
]

DRYWALL_PROMPTS = [
    "segment taping area",
    "segment joint",
    "segment drywall seam"
]

**Dataset Class**

In [ ]:
class SegDataset(Dataset):
    def __init__(self, root_dir, processor, split="train"):
        self.samples = []
        self.processor = processor

        # crack
        crack_img = os.path.join(root_dir, f"{split}_crack/images")
        crack_mask = os.path.join(root_dir, f"{split}_crack/masks")

        for f in os.listdir(crack_img):
            self.samples.append({
                "img": os.path.join(crack_img, f),
                "mask": os.path.join(crack_mask, os.path.splitext(f)[0] + ".png"),
                # "prompt": random.choice(CRACK_PROMPTS),
                "type": "crack"
            })

        # drywall
        dry_img = os.path.join(root_dir, f"{split}_drywall/images")
        dry_mask = os.path.join(root_dir, f"{split}_drywall/masks")

        for f in os.listdir(dry_img):
            self.samples.append({
                "img": os.path.join(dry_img, f),
                "mask": os.path.join(dry_mask, os.path.splitext(f)[0] + ".png"),
                # "prompt": random.choice(DRYWALL_PROMPTS),
                "type": "drywall"
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        image = Image.open(s["img"]).convert("RGB")
        mask = Image.open(s["mask"]).convert("L")

        if s["type"] == "crack":
            prompt = random.choice(CRACK_PROMPTS)
        else:
            prompt = random.choice(DRYWALL_PROMPTS)

        # processor handles image resize
        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",   # 🔥 FIX
            truncation=True         # 🔥 FIX (safe)
        )

        # match mask size with processor output
        size = inputs["pixel_values"].shape[-1]
        mask = mask.resize((size, size), resample=Image.NEAREST)

        mask = np.array(mask)
        mask = torch.tensor(mask, dtype=torch.float32)
        mask = (mask > 127).float()

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": mask,
            "type": s["type"]
        }

**Collate Function**

In [ ]:
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "type": [b["type"] for b in batch]
    }

**Load Model + Processor**

In [ ]:
processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")

**Dataset + Sampler**

In [ ]:
train_dataset = SegDataset("/kaggle/input/datasets/kartikjoshi24/segmentation-cracks-drywall/full_train/full_train", processor, "train")
val_dataset   = SegDataset("/kaggle/input/datasets/kartikjoshi24/segmentation-cracks-drywall/full_val/full_val", processor, "val")

weights = [
    5.0 if s["type"] == "drywall" else 1.0
    for s in train_dataset.samples
]

sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

**DataLoader**

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=sampler,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

**Training Setup**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

**Training Loop**

**History + tracking**

In [ ]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',     
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)


history = {
    "train_loss": [],
    "val_loss": [],
    "iou_crack": [],
    "iou_drywall": [],
    "dice_crack": [],
    "dice_drywall": [],
    "miou": [],
    "mdice": [],
    "lr": []
}

best_val_loss = float("inf")
best_miou = 0

**Training + Validation Loop**

In [ ]:
from tqdm import tqdm
lambda_ = 0.6
patience = 4
no_improve = 0

for epoch in range(20):

    print(f"\n===== Epoch {epoch+1}/20 =====")

    # ================= TRAIN =================
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        batch = {k: v.to(device) if k != "type" else v for k, v in batch.items()}

        outputs = model(
            pixel_values=batch["pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs.logits.squeeze(1)
        labels = batch["labels"]

        bce = F.binary_cross_entropy_with_logits(logits, labels)

        prob = torch.sigmoid(logits)
        
        dims = tuple(range(1, prob.ndim))
        
        intersection = (prob * labels).sum(dim=dims)
        
        dice = (2 * intersection + 1e-6) / (
            prob.sum(dim=dims) + labels.sum(dim=dims) + 1e-6
        )
        
        dice_loss = 1 - dice.mean()
        

        loss = lambda_ * bce + (1 - lambda_) * dice_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ================= VALIDATION =================
    model.eval()

    val_loss = 0

    iou_crack_sum, iou_drywall_sum = 0, 0
    dice_crack_sum, dice_drywall_sum = 0, 0

    count_crack, count_drywall = 0, 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {k: v.to(device) if k != "type" else v for k, v in batch.items()}

            outputs = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            logits = outputs.logits.squeeze(1)
            
            labels = batch["labels"]
            prob = torch.sigmoid(logits)

            bce = F.binary_cross_entropy_with_logits(logits, labels)

            dims = tuple(range(1, prob.ndim))
            
            intersection = (prob * labels).sum(dim=dims)
            
            dice = (2 * intersection + 1e-6) / (
                prob.sum(dim=dims) + labels.sum(dim=dims) + 1e-6
            )
            
            dice_loss = 1 - dice.mean()
            
            loss = lambda_ * bce + (1 - lambda_) * dice_loss

            
            val_loss += loss.item()
            preds = (prob > 0.5).float()


            for i in range(len(preds)):
                pred = preds[i]
                gt = batch["labels"][i]

                intersection = (pred * gt).sum()
                union = pred.sum() + gt.sum() - intersection

                iou = (intersection / (union + 1e-6)).item()
                dice = (2 * intersection / (pred.sum() + gt.sum() + 1e-6)).item()

                if batch["type"][i] == "crack":
                    iou_crack_sum += iou
                    dice_crack_sum += dice
                    count_crack += 1
                else:
                    iou_drywall_sum += iou
                    dice_drywall_sum += dice
                    count_drywall += 1

    # ================= METRICS =================
    val_loss /= len(val_loader)

    iou_list, dice_list = [], []

    if count_crack > 0:
        iou_crack = iou_crack_sum / count_crack
        dice_crack = dice_crack_sum / count_crack
        iou_list.append(iou_crack)
        dice_list.append(dice_crack)
    else:
        iou_crack, dice_crack = 0, 0

    if count_drywall > 0:
        iou_drywall = iou_drywall_sum / count_drywall
        dice_drywall = dice_drywall_sum / count_drywall
        iou_list.append(iou_drywall)
        dice_list.append(dice_drywall)
    else:
        iou_drywall, dice_drywall = 0, 0

    miou = sum(iou_list) / len(iou_list) if iou_list else 0
    mdice = sum(dice_list) / len(dice_list) if dice_list else 0


    # ================= HISTORY =================
    # current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["iou_crack"].append(iou_crack)
    history["iou_drywall"].append(iou_drywall)
    history["dice_crack"].append(dice_crack)
    history["dice_drywall"].append(dice_drywall)
    history["miou"].append(miou)
    history["mdice"].append(mdice)
    # history["lr"].append(current_lr)

    # ===== MODEL SELECTION (PRIMARY: mIoU, TIE-BREAK: loss) =====
    # ✅ CORRECTED ORDER:
    # 1. Save model (if improved)
    # 2. Update early stopping counter
    # 3. Check early stopping condition
    
    if (miou > best_miou) or (miou == best_miou and val_loss < best_val_loss):
        best_miou = miou
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print(f"✅ Saved BEST model | mIoU: {miou:.4f} | Val Loss: {val_loss:.4f}")
        no_improve = 0  # ✅ Reset counter when model improves
    else:
        no_improve += 1  # ✅ Increment counter when model doesn't improve

    # ================= SCHEDULER =================
    old_lr = optimizer.param_groups[0]["lr"]
    
    scheduler.step(miou)

    new_lr = optimizer.param_groups[0]["lr"]

    # ================= HISTORY =================
    history["lr"].append(new_lr)
    
    # ================= LOG =================
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"IoU Crack: {iou_crack:.4f} | Dice Crack: {dice_crack:.4f}")
    print(f"IoU Drywall: {iou_drywall:.4f} | Dice Drywall: {dice_drywall:.4f}")
    print(f"mIoU: {miou:.4f} | mDice: {mdice:.4f}")
    print(f"Learning Rate: {new_lr:.6f}")
    print(f"  No Improve:   {no_improve}/{patience}")

    # ===== EARLY STOPPING CHECK =====
    if no_improve >= patience:
        print(f"\n⛔ Early stopping triggered! No improvement for {patience} epochs.")
        print(f"   Best mIoU: {best_miou:.4f}")
        break


# ================= SAVE ONCE =================
import json
import pandas as pd

with open('training_history.json', 'w') as f:
    json.dump(history, f)

pd.DataFrame(history).to_csv("training_history.csv", index=False)

**Plotting Training History :**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# LOAD HISTORY
# -----------------------------
history = pd.read_csv("/kaggle/working/training_history.csv")

epochs = range(1, len(history) + 1)

sns.set(style="whitegrid")

# -----------------------------
# 1. LOSS CURVE
# -----------------------------
plt.figure(figsize=(8,5))
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# -----------------------------
# 2. mIoU & mDice
# -----------------------------
plt.figure(figsize=(8,5))
plt.plot(epochs, history["miou"], label="mIoU")
plt.plot(epochs, history["mdice"], label="mDice")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Overall Segmentation Performance")
plt.legend()
plt.show()

# -----------------------------
# 3. PER-CLASS IoU
# -----------------------------
plt.figure(figsize=(8,5))
plt.plot(epochs, history["iou_crack"], label="IoU Crack")
plt.plot(epochs, history["iou_drywall"], label="IoU Drywall")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("Per-Class IoU")
plt.legend()
plt.show()

# -----------------------------
# 4. PER-CLASS DICE
# -----------------------------
plt.figure(figsize=(8,5))
plt.plot(epochs, history["dice_crack"], label="Dice Crack")
plt.plot(epochs, history["dice_drywall"], label="Dice Drywall")
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Per-Class Dice")
plt.legend()
plt.show()

# -----------------------------
# 5. LEARNING RATE
# -----------------------------
plt.figure(figsize=(8,5))
plt.plot(epochs, history["lr"], label="Learning Rate")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.title("Learning Rate Schedule")
plt.legend()
plt.show()

**Model 1 Infrence :**

In [ ]:
import os
import cv2
import torch
import numpy as np
import shutil
from PIL import Image
from tqdm import tqdm

model.eval()

OUTPUT_DIR = "/kaggle/working/predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROMPTS = {
    "crack": "segment crack",
    "drywall": "segment drywall seam"
}

with torch.no_grad():
    for sample in tqdm(val_dataset.samples):

        img_path = sample["img"]
        img_type = sample["type"]

        image = Image.open(img_path).convert("RGB")
        orig_w, orig_h = image.size

        prompt = PROMPTS[img_type]

        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        logits = outputs.logits.squeeze(1)

        pred = torch.sigmoid(logits)[0].cpu().numpy()

        # binarize
        pred = (pred > 0.5).astype(np.uint8) * 255

        # resize back
        pred = cv2.resize(pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

        # ensure binary after resize
        pred[pred > 0] = 255

        # filename
        img_name = os.path.basename(img_path)
        img_id = os.path.splitext(img_name)[0]

        safe_prompt = prompt.lower().replace(" ", "_")
        filename = f"{img_id}__{safe_prompt}.png"

        cv2.imwrite(os.path.join(OUTPUT_DIR, filename), pred.astype(np.uint8))

# ---------------- ZIP ----------------
ZIP_PATH = "/kaggle/working/predictions.zip"

shutil.make_archive(
    ZIP_PATH.replace(".zip", ""),
    'zip',
    OUTPUT_DIR
)

print("✅ ZIP created:", ZIP_PATH)

**Inference Time calculation :**

In [ ]:
import time
import torch

num_images = len(val_dataset.samples)

model_time = 0.0

if torch.cuda.is_available():
    torch.cuda.synchronize()

with torch.no_grad():
    for sample in val_dataset.samples:

        image = Image.open(sample["img"]).convert("RGB")
        prompt = PROMPTS[sample["type"]]

        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        # ---------------- MODEL TIMING ONLY ----------------
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.time()

        outputs = model(**inputs)
        logits = outputs.logits.squeeze(1)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.time()

        model_time += (end - start)

# ---------------- RESULTS ----------------
avg_model_time = model_time / num_images
fps = 1 / avg_model_time

print(f"⚡ Pure Model Time: {avg_model_time:.4f} sec/image")
print(f"🚀 Model FPS: {fps:.2f}")

**model size :**

In [ ]:
# import os
# print(os.path.getsize("best_model.pth") / 1e6, "MB")

**visual_eg :**

In [ ]:
import matplotlib.pyplot as plt
import random
import cv2
import numpy as np
from PIL import Image

random.seed(42)

# -----------------------------
# SEPARATE SAMPLES BY TYPE
# -----------------------------
crack_samples = [s for s in val_dataset.samples if s["type"] == "crack"]
drywall_samples = [s for s in val_dataset.samples if s["type"] == "drywall"]

samples = random.sample(crack_samples, 3) + random.sample(drywall_samples, 3)
random.shuffle(samples)

# -----------------------------
# PLOTTING
# -----------------------------
plt.figure(figsize=(15, 12))

for i, s in enumerate(samples):

    img = Image.open(s["img"]).convert("RGB")
    gt = Image.open(s["mask"]).convert("L")

    # ensure binary GT
    gt = np.array(gt)
    gt = (gt > 127).astype(np.uint8) * 255

    prompt = "segment crack" if s["type"] == "crack" else "segment drywall seam"

    inputs = processor(
        text=prompt,
        images=img,
        return_tensors="pt",
        padding="max_length",
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(1)
        pred = torch.sigmoid(logits)[0].cpu().numpy()

    pred = (pred > 0.5).astype(np.uint8) * 255
    pred = cv2.resize(pred, img.size, interpolation=cv2.INTER_NEAREST)
    pred[pred > 0] = 255

    row = i + 1

    plt.subplot(6, 3, 3*row - 2)
    plt.imshow(img)
    plt.title(f"{s['type'].capitalize()} | Prompt: '{prompt}'")
    plt.axis("off")

    plt.subplot(6, 3, 3*row - 1)
    plt.imshow(gt, cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(6, 3, 3*row)
    plt.imshow(pred, cmap="gray")
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.subplots_adjust(hspace=0.3)

# ✅ SAVE BEFORE SHOW
plt.savefig("visual_examples.png", dpi=300, bbox_inches="tight")
plt.show()